# Cálculo del AP en el test de FloorPlanCAD

> **Qué compara.** El AP50 de seg de pared da 0.82 en el test de FloorPlanCAD y 0.12 en BLD-AR, pero los dos números no se calculan igual. Este notebook evalúa el mismo modelo sobre los mismos planos, los del test de FloorPlanCAD, con cuatro cálculos. Cada uno cambia una sola cosa respecto del anterior, desde el cálculo del entrenamiento hasta el de BLD-AR. Así, la diferencia entre dos cálculos seguidos muestra cuánto AP se pierde solo por esa forma de calcular, sin que cambie el modelo ni los planos.
>
> | Cálculo | Qué cambia respecto del anterior | Qué intenta explicar |
> |---|---|---|
> | Ultralytics | Nada: es el cálculo del entrenamiento, el que da 0.82 | Punto de partida |
> | (3) Sin unir, máscaras a 1/4 | La herramienta (COCOeval en vez de Ultralytics), la confianza mínima (0.05) y la inferencia con SAHI. Las máscaras se siguen comparando a 1/4 del tamaño de la imagen, como hace Ultralytics | Cuánto cambia el AP solo por la herramienta |
> | (2) Sin unir | Las máscaras se comparan con todo el detalle del plano, como en BLD-AR | Cuánto cuesta comparar con más detalle: en un muro fino, un corrimiento chico baja más la superposición |
> | (1) Tal cual BLD-AR | Se aplica la unión de predicciones de SAHI, que junta las predicciones cuyas cajas se superponen | Cuánto cuesta la unión |
>
> Los planos del test miden ~1 000 px y entran en un solo tile: acá no hay muros cortados entre tiles, así que todo lo que junte la unión son paredes distintas. Se evalúan las 7 clases que se comparan con BLD-AR (`docs/MAPEO_CLASES.md`).

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8 pycocotools==2.0.11

## Imports

In [ ]:
import math
import tempfile
from pathlib import Path

import numpy as np
import yaml
from IPython.display import Markdown, display
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel
from sahi.postprocess.combine import GreedyNMMPostprocess
from sahi.predict import get_prediction
from ultralytics import YOLO

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
FLOORPLANCAD = REPO / "datasets/floor-plan-cad-yolo"
DATA = FLOORPLANCAD / "floor-plan-cad-yolo.yaml"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024        # tile de la evaluación en BLD-AR, igual al imgsz de la corrida
CONF = 0.05        # como en la evaluación en BLD-AR
MASK_RATIO = 4     # Ultralytics compara las máscaras a 1/4 del imgsz
DEVICE = "cuda:0"

# Clases que se comparan con BLD-AR (docs/MAPEO_CLASES.md)
CATEGORIES = {name: i + 1 for i, name in enumerate(
    ["wall", "single_door", "window", "sliding_door", "stair", "elevator", "railing"])}

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Referencia: cálculo de Ultralytics

> Igual que la evaluación en test del notebook de entrenamiento. Los resultados se guardan en una carpeta temporal.

In [ ]:
yolo = YOLO(WEIGHTS)
metrics = yolo.val(data=str(DATA), split="test", imgsz=TILE, batch=2, device=DEVICE,
                   project=tempfile.mkdtemp(), plots=False, verbose=False)
ultralytics = {yolo.names[c]: (metrics.seg.ap50[i], metrics.seg.ap[i])
               for i, c in enumerate(metrics.seg.ap_class_index) if yolo.names[c] in CATEGORIES}

## Etiquetas del test en formato COCO

In [ ]:
def coco_annotation(points, image_id, category_id):
    x, y = points[:, 0], points[:, 1]
    return {
        "image_id": image_id, "category_id": category_id, "iscrowd": 0,
        "segmentation": [points.ravel().tolist()],
        "bbox": [float(x.min()), float(y.min()), float(x.max() - x.min()), float(y.max() - y.min())],
        "area": float(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))),
    }


names = yaml.safe_load(DATA.read_text())["names"]
image_paths = sorted((FLOORPLANCAD / "images/test").iterdir())
images, annotations = [], []
for image_id, image_path in enumerate(image_paths, start=1):
    width, height = Image.open(image_path).size
    images.append({"id": image_id, "width": width, "height": height})
    for line in (FLOORPLANCAD / "labels/test" / f"{image_path.stem}.txt").read_text().splitlines():
        row = line.split()
        name = names[int(row[0])]
        if name in CATEGORIES:
            # Etiqueta YOLO: clase y puntos x, y entre 0 y 1
            points = np.reshape(np.array(row[1:], dtype=float), (-1, 2)) * (width, height)
            annotations.append(coco_annotation(points, image_id, CATEGORIES[name]))
for annotation_id, annotation in enumerate(annotations, start=1):
    annotation["id"] = annotation_id

## Modelo

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Predicciones con y sin unir

> Con un solo tile, la inferencia por tiles de la evaluación en BLD-AR es la predicción del plano entero seguida de la unión (`get_sliced_prediction` de SAHI). Acá se separan los dos pasos: las mismas predicciones se evalúan sin unir y unidas, así que la diferencia es solo la unión.

In [ ]:
# Misma unión que la evaluación en BLD-AR: los valores por defecto de get_sliced_prediction
merge = GreedyNMMPostprocess(match_threshold=0.5, match_metric="IOS", class_agnostic=False)


def to_coco(object_predictions, image_id):
    """Predicciones en formato COCO, solo de las clases evaluadas."""
    predictions = []
    for object_prediction in object_predictions:
        prediction = object_prediction.to_coco_prediction(image_id=image_id).json
        if prediction["category_name"] in CATEGORIES:
            prediction["category_id"] = CATEGORIES[prediction["category_name"]]
            predictions.append(prediction)
    return predictions


unmerged, merged = [], []
for image, image_path in zip(images, image_paths):
    object_predictions = get_prediction(str(image_path), model).object_prediction_list
    unmerged += to_coco(object_predictions, image["id"])
    merged += to_coco(merge(list(object_predictions)), image["id"])
print(f"{len(unmerged)} predicciones sin unir y {len(merged)} unidas en {len(images)} planos")

## Métricas

> Para comparar a 1/4 se achican las etiquetas y las predicciones al tamaño de las máscaras de Ultralytics (imgsz / 4 = 256 px de lado).

In [ ]:
def evaluate(images, annotations, predictions):
    """AP50 y AP50-95 de seg por clase, con COCOeval como en la evaluación en BLD-AR."""
    coco_gt = COCO()
    coco_gt.dataset = {"images": images, "annotations": annotations,
                       "categories": [{"id": i, "name": name} for name, i in CATEGORIES.items()]}
    coco_gt.createIndex()
    # Copias: COCOeval reemplaza los polígonos de las predicciones por RLE
    evaluator = COCOeval(coco_gt, coco_gt.loadRes([dict(p) for p in predictions]), "segm")
    evaluator.evaluate()
    evaluator.accumulate()
    precision = evaluator.eval["precision"][:, :, :, 0, -1]  # IoU x recall x clase (área "all", 100 detecciones)
    mean = lambda p: p[p > -1].mean() if (p > -1).any() else float("nan")
    return {name: (mean(precision[0, :, k]), mean(precision[:, :, k])) for k, name in enumerate(CATEGORIES)}


factor = {image["id"]: TILE / max(image["width"], image["height"]) / MASK_RATIO for image in images}


def reduced(item):
    """Copia de una etiqueta o predicción con las coordenadas al tamaño de las máscaras de Ultralytics."""
    k = factor[item["image_id"]]
    return {**item, "segmentation": [(np.array(polygon) * k).tolist() for polygon in item["segmentation"]],
            "bbox": [v * k for v in item["bbox"]], "area": item["area"] * k * k}


small_images = [{**image, "width": math.ceil(image["width"] * factor[image["id"]]),
                 "height": math.ceil(image["height"] * factor[image["id"]])} for image in images]
results = {
    "Ultralytics": ultralytics,
    "(3) Sin unir, máscaras a 1/4": evaluate(small_images, [reduced(a) for a in annotations], [reduced(p) for p in unmerged]),
    "(2) Sin unir": evaluate(images, annotations, unmerged),
    "(1) Tal cual BLD-AR": evaluate(images, annotations, merged),
}

## Resultado

> Cada columna cambia una sola cosa respecto de la anterior: la diferencia entre dos columnas seguidas es lo que cuesta ese cambio. El promedio es el de las 7 clases.

In [ ]:
instances = {name: sum(a["category_id"] == i for a in annotations) for name, i in CATEGORIES.items()}
value = lambda result, name, k: result.get(name, (float("nan"),) * 2)[k]

for k, metric in enumerate(["AP50", "AP50-95"]):
    table = (f"**{metric} de seg · test de FloorPlanCAD · {WEIGHTS.parent.name}**\n\n"
             "| Clase | Instancias | " + " | ".join(results) + " |\n|---|---|" + "---|" * len(results) + "\n")
    for name in CATEGORIES:
        table += f"| {name} | {instances[name]} | " + " | ".join(f"{value(r, name, k):.3f}" for r in results.values()) + " |\n"
    table += "| Promedio | | " + " | ".join(
        f"{np.nanmean([value(r, name, k) for name in CATEGORIES]):.3f}" for r in results.values()) + " |"
    display(Markdown(table))

wall = [value(r, "wall", 0) for r in results.values()]
display(Markdown(
    "**AP50 de seg de pared que se pierde en cada paso**\n\n| Paso | Antes | Después | Diferencia |\n|---|---|---|---|\n"
    + "".join(f"| {step} | {before:.3f} | {after:.3f} | {after - before:+.3f} |\n" for step, before, after in zip(
        ["Herramienta: Ultralytics → (3)", "Resolución de las máscaras: (3) → (2)", "Unión de predicciones: (2) → (1)"],
        wall, wall[1:]))))